## MonReader - part 5

----

### Generative Text-to-Speech with Sesame (CSM)

**Objective.**  
Explore and integrate **Sesame (Conversational Speech Model – CSM)** as a **generative Text-to-Speech engine** for long-form literary prose, building on the **canonical, sentence-aware chunks** produced in **MonReader – Part 4**.

We continue using the same two books:
- *The Chamber* — John Grisham *(English)*
- *A onda que se ergueu no mar* — Ruy Castro *(Portuguese)*

This notebook is intentionally **hands-on and exploratory**: we focus on understanding Sesame’s **input expectations**, **GPU inference workflow** (PyTorch / Hugging Face), and **stability on narrative text**, rather than benchmarking multiple TTS systems.

### Why Sesame?

Sesame represents a newer class of **generative speech models**, aiming for natural pacing and expressive prosody by modeling speech more holistically than classic TTS pipelines.

We explore Sesame because it promises:
- **Natural prosody and rhythm**
- **Better long-form behavior** than many traditional TTS stacks
- A research-friendly setup for inspection and experimentation

### What We Do in This Part

- Set up the environment and verify **GPU execution**
- Run minimal synthesis to learn the **API + audio outputs**
- Test controlled samples from both books (English vs. Portuguese, short vs. long chunks)
- Record observations and define a clean **integration path** back into MonReader

> The goal here is correctness and understanding: learn how to use Sesame reliably for long-form audiobook-style synthesis, and identify its practical limits early.


---

## Step J.1 — Environment Setup + GPU Verification

This step ensures:
- packages are available (Transformers w/ CSM support)
- GPU is visible to PyTorch (CUDA)
- we capture a reproducible environment report

In [1]:
from pathlib import Path
import os
import sys
import platform
import subprocess
import json
import time

In [2]:
BASE = Path.cwd()
WORK_DIR = BASE / "work"

# Part 5 outputs
STEP5_DIR = WORK_DIR / "step5_sesame_csm"
STEP5_DIR.mkdir(parents=True, exist_ok=True)


In [3]:
# Verify PyTorch and CUDA version

import torch, transformers
from transformers import AutoProcessor, CsmForConditionalGeneration
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Transformers:", transformers.__version__)


e:\Devs\pyEnv-1\venvs\MonReader_env_part5\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch: 2.5.1+cu121
CUDA available: True
Transformers: 4.57.3


Run `huggingface-cli login` on the terminal:

```

    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|
``` 

```
A token is already saved on your machine. Run `hf auth whoami` to get more information or `hf auth logout` if you want to log out.
Setting a new token will erase the existing one.
To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Token can be pasted using 'Right-Click'.

Enter your token (input will not be visible):
Add token as git credential? (Y/n) Y
Token is valid (permission: fineGrained).
```


### Load Sesame model + processor

This will download weights on first run (~1B params)

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [ ]:
MODEL_ID = "sesame/csm-1b"

print("Loading processor...")
processor = AutoProcessor.from_pretrained(MODEL_ID)

print("Loading model...")
model = CsmForConditionalGeneration.from_pretrained(
    MODEL_ID,
    #torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    dtype=torch.float16 if device == "cuda" else torch.float32,
)
model = model.to(device)
model.eval()

print("Model loaded.")


Loading processor...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading model...


Loading checkpoint shards: 100%|██████████| 2/2 [00:19<00:00,  9.85s/it]


Model loaded.


### Sesame (CSM) — Model Overview and Architecture

#### What is Sesame (CSM)?

**Sesame (Conversational Speech Model, CSM)** is a **fully generative text-to-speech model** that produces natural-sounding speech by modeling audio as a sequence of discrete tokens, rather than relying on a traditional multi-stage TTS pipeline.

Unlike classical TTS systems (text normalization → phonemes → acoustic model → vocoder), Sesame operates as a **single, end-to-end generative model**, conceptually similar to modern large language models, but trained to emit **audio tokens instead of text tokens**.

In the MonReader project, Sesame is used as the **final synthesis stage**, converting clean, sentence-aware OCR text into audiobook-style speech.

---

### The Processor

Example initialization:

    processor = AutoProcessor.from_pretrained("sesame/csm-1b")

The **processor** is a Hugging Face abstraction that encapsulates **all non-learned input and output transformations** required by the model.

For Sesame, the processor fulfills two essential roles:

#### 1. Text preprocessing (input side)

- Accepts raw Unicode text directly (no phoneme conversion required)
- Tokenizes text into a sequence of **text tokens** compatible with the model’s embedding space
- Applies internal normalization consistent with the model’s training configuration

This design allows Sesame to operate directly on canonical prose without language-specific text normalization pipelines.

#### 2. Audio decoding (output side)

- Converts generated **discrete audio tokens** into a continuous waveform
- Uses an internal **neural audio codec (Mimi)** for reconstruction
- Outputs PCM audio as a NumPy array at the correct sampling rate

The processor therefore acts as the **bridge between symbolic text and audible waveform data**.

---

### The Model

Example initialization:

    model = CsmForConditionalGeneration.from_pretrained("sesame/csm-1b")

The **model** is a large-scale neural network (≈1 billion parameters) trained for **conditional sequence generation**, where:

- **Input**: a sequence of text tokens
- **Output**: a sequence of discrete audio tokens

The model itself does not emit waveforms; instead, it predicts audio tokens that are later decoded by the processor.

---

### High-Level Architecture

At a high level, Sesame follows a **Transformer-based encoder–decoder architecture**, adapted for speech generation.

#### Conceptual layout

    Text
     │
     ▼
    Text Tokenizer
     │
     ▼
    Text Embeddings
     │
     ▼
    Transformer Encoder
     │
     ▼
    Cross-Attention
     │
     ▼
    Transformer Decoder
     │
     ▼
    Discrete Audio Tokens
     │
     ▼
    Neural Audio Codec (Mimi)
     │
     ▼
    Waveform (PCM audio)

---

### Key Technical Components

#### Transformer backbone

- Multi-layer Transformer architecture
- Self-attention for long-range dependency modeling
- Cross-attention between text and audio token streams
- Supports coherent prosody across long sentences and paragraphs

#### Discrete audio token modeling

- Speech is represented as sequences of **discrete latent codes**
- These codes correspond to compressed representations learned by the Mimi codec
- This reduces generation complexity compared to raw waveform prediction

#### Neural audio codec (Mimi)

- Learns a compact latent space for speech audio
- Enables high perceptual quality and stable reconstruction
- Decoding is performed after token generation, not during Transformer inference

#### Autoregressive generation

- Audio tokens are generated sequentially
- Output duration is controlled via `max_new_tokens`
- Prosody, pauses, and rhythm emerge from learned patterns rather than explicit rules

---

### Why Sesame Fits MonReader

Sesame aligns well with MonReader’s design goals:

- Works directly on **raw, canonical prose**
- Exhibits **long-form stability** suitable for audiobook-style narration
- Produces **natural prosody and pacing**
- Avoids complex, language-specific TTS pipelines

This supports MonReader’s guiding principle:

> Minimal intervention, maximum narrative fidelity.

---

### Inference Characteristics

- GPU-accelerated (FP16 recommended)
- First inference downloads model shards (~1B parameters)
- Generation cost scales primarily with audio length
- Memory usage dominated by Transformer layers rather than codec decoding

---



### Step J.2 — Minimal Sesame synthesis (sanity check)

Goal:
- Generate a single WAV file from a short sentence
- Save it into `work/step5_sesame_csm/`
- Play it back inside the notebook

In [6]:
import numpy as np
import soundfile as sf
from IPython.display import Audio, display

#### Step J.2.1 - minimal helper for synthesis -> WAV file


In [7]:
def sesame_tts_to_wav(
    text: str,
    wav_path: Path,
    speaker_id: int = 0,
    max_length: int = 1024,
    do_sample: bool = False,
    temperature: float = 0.7,
):
    """
    Minimal, reproducible TTS call for Sesame (CSM).
    Saves a WAV file and returns (wav_path, sampling_rate).
    """
    wav_path = Path(wav_path)
    wav_path.parent.mkdir(parents=True, exist_ok=True)

    # CSM commonly uses speaker tags like [0], [1], ...
    formatted = f"[{speaker_id}]{text}".strip()

    # Build inputs
    inputs = processor(formatted, add_special_tokens=True, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Generate audio (output_audio=True makes model.generate return audio)
    with torch.no_grad():
        audio = model.generate(
            **inputs,
            output_audio=True,
            max_length=max_length,
            do_sample=do_sample,
            temperature=temperature,
        )

    # Save audio using the processor utility (recommended)
    processor.save_audio(audio, str(wav_path))

    # Retrieve sampling rate from processor
    sr = processor.feature_extractor.sampling_rate
    return wav_path, sr


#### Step J.2.2 - run one short test sentence


In [8]:
test_text = "This is a short test of the Sesame text to speech model."
out_wav = STEP5_DIR / "sesame_test_en.wav"

wav_path, sr = sesame_tts_to_wav(
    text=test_text,
    wav_path=out_wav,
    speaker_id=0,
    max_length=1024,
    do_sample=False,   # deterministic baseline
)

print("Saved:", wav_path)
print("Sampling rate:", sr)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Both `max_new_tokens` (=125) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Saved: e:\Devs\pyEnv-1\Apziva\MonReader\work\step5_sesame_csm\sesame_test_en.wav
Sampling rate: 24000


#### Step J.2.3 - Playback inside notebook

In [9]:
# Load and play
audio_np, sr = sf.read(wav_path)
display(Audio(audio_np, rate=sr))

#### Step J.3 - Sampling vs Determinism (prosody control)

#### J.3.1 - Generate a small "sampling grid"

We synthesize the same sentence with different sampling settings to observe changes in prosody, pacing, and emphasis.


In [10]:
import pandas as pd

In [11]:
grid_text = "The judge paused for a moment, then spoke slowly and clearly."

configs = [
    {"tag": "deterministic", "do_sample": False, "temperature": 0.7},
    {"tag": "sample_t06",    "do_sample": True,  "temperature": 0.6},
    {"tag": "sample_t09",    "do_sample": True,  "temperature": 0.9},
]


In [12]:
rows = []
for cfg in configs:
    out_wav = STEP5_DIR / f"grid_en_{cfg['tag']}.wav"
    wav_path, sr = sesame_tts_to_wav(
        text=grid_text,
        wav_path=out_wav,
        speaker_id=0,
        max_length=1024,
        do_sample=cfg["do_sample"],
        temperature=cfg["temperature"],
    )
    audio_np, _ = sf.read(wav_path)
    duration_s = len(audio_np) / sr

    rows.append({
        "exp": "sampling_grid",
        "tag": cfg["tag"],
        "do_sample": cfg["do_sample"],
        "temperature": cfg["temperature"],
        "wav_path": str(wav_path),
        "sr": sr,
        "duration_s": round(duration_s, 2),
        "text": grid_text,
    })

df_sampling = pd.DataFrame(rows)
df_sampling


Both `max_new_tokens` (=125) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=125) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=125) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


,exp,tag,do_sample,temperature,wav_path,sr,duration_s,text
0,sampling_grid,deterministic,False,0.7,e:\Devs\pyEnv-1\Apziva\MonReader\work\step5_se...,24000,3.20,"The judge paused for a moment, then spoke slow..."
1,sampling_grid,sample_t06,True,0.6,e:\Devs\pyEnv-1\Apziva\MonReader\work\step5_se...,24000,6.48,"The judge paused for a moment, then spoke slow..."
2,sampling_grid,sample_t09,True,0.9,e:\Devs\pyEnv-1\Apziva\MonReader\work\step5_se...,24000,2.64,"The judge paused for a moment, then spoke slow..."


#### J.3.2 - Quick playback (one by one)

In [13]:
from IPython.display import Markdown

In [14]:
for _, r in df_sampling.iterrows():
    display(Markdown(f"**{r['tag']}** | do_sample={r['do_sample']} | temp={r['temperature']} | dur={r['duration_s']}s"))
    audio_np, sr = sf.read(r["wav_path"])
    display(Audio(audio_np, rate=sr))


**deterministic** | do_sample=False | temp=0.7 | dur=3.2s

**sample_t06** | do_sample=True | temp=0.6 | dur=6.48s

**sample_t09** | do_sample=True | temp=0.9 | dur=2.64s

#### Step J.4 - Punctuation stress test (OCR-relevant)

#### J.4.1 - Three punctuation variants

Same content with different punctuation to observe pause control and prosody.


In [15]:
base = "He opened the letter, read it twice, and then whispered: I knew it."
variants = {
    "punct_normal": base,
    "punct_removed": "He opened the letter read it twice and then whispered I knew it",
    "punct_heavy": "He opened the letter... read it twice... and then whispered: 'I knew it.'",
}


In [16]:
rows = []
for tag, txt in variants.items():
    out_wav = STEP5_DIR / f"punct_en_{tag}.wav"
    wav_path, sr = sesame_tts_to_wav(
        text=txt,
        wav_path=out_wav,
        speaker_id=0,
        max_length=1024,
        do_sample=False,
    )
    audio_np, _ = sf.read(wav_path)
    duration_s = len(audio_np) / sr

    rows.append({
        "exp": "punctuation",
        "tag": tag,
        "wav_path": str(wav_path),
        "duration_s": round(duration_s, 2),
        "text": txt,
    })

df_punct = pd.DataFrame(rows)
df_punct


Both `max_new_tokens` (=125) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=125) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=125) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


,exp,tag,wav_path,duration_s,text
0,punctuation,punct_normal,e:\Devs\pyEnv-1\Apziva\MonReader\work\step5_se...,3.76,"He opened the letter, read it twice, and then ..."
1,punctuation,punct_removed,e:\Devs\pyEnv-1\Apziva\MonReader\work\step5_se...,10.00,He opened the letter read it twice and then wh...
2,punctuation,punct_heavy,e:\Devs\pyEnv-1\Apziva\MonReader\work\step5_se...,10.00,He opened the letter... read it twice... and t...


In [17]:
for _, r in df_punct.iterrows():
    display(Markdown(f"**{r['tag']}** | dur={r['duration_s']}s"))
    audio_np, sr = sf.read(r["wav_path"])
    display(Audio(audio_np, rate=sr))

**punct_normal** | dur=3.76s

**punct_removed** | dur=10.0s

**punct_heavy** | dur=10.0s

#### Step J.5 - English vs Portuguese vs mixed (multilingual probe)

We test pronunciation and stability across languages.

In [18]:
samples = {
    "en": "The courtroom was silent as the verdict was read aloud.",
    "pt": "A sala do tribunal ficou em silêncio quando a sentença foi lida em voz alta.",
    "mix": "The courtroom was silent. A sala do tribunal ficou em silêncio.",
}

In [19]:
rows = []
for tag, txt in samples.items():
    out_wav = STEP5_DIR / f"lang_{tag}.wav"
    wav_path, sr = sesame_tts_to_wav(
        text=txt,
        wav_path=out_wav,
        speaker_id=0,
        max_length=2048,
        do_sample=False,
    )
    audio_np, _ = sf.read(wav_path)
    duration_s = len(audio_np) / sr

    rows.append({
        "exp": "multilingual",
        "tag": tag,
        "duration_s": round(duration_s, 2),
        "wav_path": str(wav_path),
        "text": txt,
    })

df_lang = pd.DataFrame(rows)
df_lang


Both `max_new_tokens` (=125) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=125) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=125) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


,exp,tag,duration_s,wav_path,text
0,multilingual,en,10.00,e:\Devs\pyEnv-1\Apziva\MonReader\work\step5_se...,The courtroom was silent as the verdict was re...
1,multilingual,pt,10.00,e:\Devs\pyEnv-1\Apziva\MonReader\work\step5_se...,A sala do tribunal ficou em silêncio quando a ...
2,multilingual,mix,3.68,e:\Devs\pyEnv-1\Apziva\MonReader\work\step5_se...,The courtroom was silent. A sala do tribunal f...


In [20]:
for _, r in df_lang.iterrows():
    display(Markdown(f"**{r['tag']}** | dur={r['duration_s']}s"))
    audio_np, sr = sf.read(r["wav_path"])
    display(Audio(audio_np, rate=sr))

**en** | dur=10.0s

**pt** | dur=10.0s

**mix** | dur=3.68s

----

#### Step J.6 — “Peek at tokens” (text → audio codes)

##### J.6.1 — Inspect the text tokens



In [21]:
# Text tokens
probe_text = "[0]The past is just a story we tell ourselves."
tok = processor(probe_text, add_special_tokens=True, return_tensors="pt")
input_ids = tok["input_ids"][0]  # (seq_len,)

print("Text tokens (ids):", input_ids.tolist()[:32], "...")
print("Sequence length:", input_ids.shape[0])

# quick sanity: decode back (ignoring non-printables)
if hasattr(processor, "tokenizer"):
    print("Round-trip decode (first 80 chars):",
          processor.tokenizer.decode(input_ids[:80], skip_special_tokens=False))


Text tokens (ids): [128000, 58, 15, 60, 791, 3347, 374, 1120, 264, 3446, 584, 3371, 13520, 13, 128001] ...
Sequence length: 15
Round-trip decode (first 80 chars): <|begin_of_text|>[0]The past is just a story we tell ourselves.<|end_of_text|>


##### J.6.2 — Inspect the audio tokens for your generated speech

CSM’s generate(..., output_audio=True) gives you waveform. To see the discrete tokens that the codec uses, a practical workaround is to re-encode the generated audio with Mimi (the same codec family CSM uses under the hood). That gives you a grid of RVQ codebook tokens: shape (num_codebooks, frames), typically 32 codebooks for CSM.

In [22]:
# J.6.2 Audio tokens via Mimi
from transformers import MimiModel, AutoFeatureExtractor
import soundfile as sf
import torch
import numpy as np


In [23]:
# 1) Use our existing helper to synthesize a short utterance
text_for_audio = "The judge paused for a moment, then spoke slowly and clearly."
wav_path, sr = sesame_tts_to_wav(
    text=text_for_audio,
    wav_path=STEP5_DIR / "tokens_probe.wav",
    speaker_id=0,
    max_length=1024,
    do_sample=False,
)

# 2) Load waveform
audio_np, sr = sf.read(wav_path)
audio_np = np.asarray(audio_np, dtype=np.float32)

# 3) Load Mimi and its feature extractor
mimi_id = "kyutai/mimi"
mimi = MimiModel.from_pretrained(mimi_id)
mimi.eval()
fe = AutoFeatureExtractor.from_pretrained(mimi_id)

# 4) Prepare inputs for Mimi encoder
inputs = fe(raw_audio=audio_np, sampling_rate=fe.sampling_rate, return_tensors="pt")

with torch.no_grad():
    enc = mimi.encode(inputs["input_values"], inputs.get("padding_mask"))

# enc.audio_codes: (batch=1, num_quantizers, codes_length)
codes = enc.audio_codes[0].cpu().numpy()  # (Q, T)
Q, T = codes.shape
print(f"Mimi codes shape: (num_codebooks={Q}, frames={T})")
print("First frame across codebooks:", codes[:, 0].tolist()[:16], "...")  # first 16 codebooks
print("Token grid (top-left 6x10):")
print(codes[:6, :10])  # small slice


Both `max_new_tokens` (=125) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Mimi codes shape: (num_codebooks=32, frames=125)
First frame across codebooks: [995, 929, 552, 643, 1951, 182, 168, 351, 336, 2031, 1176, 140, 170, 1631, 383, 996] ...
Token grid (top-left 6x10):
[[ 995  359  908 1041 1178  327 1056  653  306   73]
 [ 929 1028  665 1396 1065 1312 1505 1624  855 1690]
 [ 552  308  385   22 1686  778  723 1663  460  653]
 [ 643  428 1863  642 1293 1478 1507  994 1309  536]
 [1951  436  565 1152  825 1641  206 1614 1383   20]
 [ 182 1716 2047  146  739 1664 1299 1413 1035 1014]]
